# MACD-V (Volatility-Normalized MACD) Strategy Notebook

**Framework:** Rick's FTMO Strategy Testing Template  
**Indicator:** MACD-V — MACD normalized by rolling volatility  
**Purpose:** Grid search optimization with IS/OOS validation and sensitivity analysis  

---

### What is MACD-V?
MACD-V adjusts the traditional MACD line by current market volatility to:
1. Prevent false signals during low-volatility consolidation
2. Scale signals appropriately during high-volatility trends
3. Improve signal quality across different market regimes

### Signal Logic
- **LONG Entry:** MACD-V histogram crosses from negative to positive
- **LONG Exit:** MACD-V histogram crosses from positive to negative

In [ ]:
# !pip install yfinance
# !pip install TA-Lib 
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from itertools import product

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

---
## Section 1: Data Loading
Download BTC-USD daily data from Yahoo Finance (2018 to present)

In [ ]:
# DOWNLOAD STOCK DATA FROM 2018 USING YFINANCE

# Configuration - Change these variables as needed
TICKER = 'BTC-USD'  # Any ticker symbol (e.g., 'EURUSD=X', 'GC=F', '^GSPC')
START_DATE = '2018-01-01'  # Any start date in YYYY-MM-DD format

# Download data from start date onwards
try:
    stock_data = yf.download(TICKER, start=START_DATE, interval='1d')
except Exception as e:
    print(f"Error downloading data: {e}")
    stock_data = pd.DataFrame()

if not stock_data.empty:
    print(f"Successfully downloaded {len(stock_data)} records for {TICKER} from {START_DATE}")
    print(f"Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")
    print("\nFirst 5 rows:")
    print(stock_data.head())
else:
    print(f"Failed to download {TICKER} data from yfinance")

# Display the downloaded data
stock_data

---
## Section 2: Indicator Calculation — MACD-V

**Steps:**
1. Calculate traditional MACD components via TA-Lib
2. Compute exponentially weighted rolling volatility of returns
3. Normalize MACD line by volatility → MACD-V
4. Recalculate signal line on the normalized MACD-V
5. Compute MACD-V histogram

In [ ]:
# MACD-V INDICATOR CALCULATION

if "stock_data" not in locals() or stock_data.empty:
    raise ValueError("Please run the stock data download cell first")

# Extract close price (handling multi-level columns from yfinance)
if isinstance(stock_data.columns, pd.MultiIndex):
    close_arr = stock_data[("Close", TICKER)].values
    high_arr = stock_data[("High", TICKER)].values
    low_arr = stock_data[("Low", TICKER)].values
    volume_arr = stock_data[("Volume", TICKER)].values
else:
    close_arr = stock_data["Close"].values
    high_arr = stock_data["High"].values
    low_arr = stock_data["Low"].values
    volume_arr = stock_data["Volume"].values

print(f"Calculating MACD-V indicators for {TICKER}...")

# Step 1: Traditional MACD (default params for display)
macd_line, signal_line, macd_hist = talib.MACD(close_arr, fastperiod=12, slowperiod=26, signalperiod=9)

# Step 2: Rolling volatility of returns (EWM std, span=20)
close_series = pd.Series(close_arr, index=stock_data.index)
returns = close_series.pct_change()
volatility = returns.ewm(span=20).std()

# Step 3: Normalize MACD by volatility
macd_v_line = pd.Series(macd_line, index=stock_data.index) / volatility

# Step 4: Signal on normalized MACD-V
macd_v_signal = macd_v_line.ewm(span=9).mean()

# Step 5: MACD-V histogram
macd_v_hist = macd_v_line - macd_v_signal

# Build indicators dataframe for display
indicators_df = pd.DataFrame({
    "Date": stock_data.index,
    "Close": close_arr,
    "MACD_Line": macd_line,
    "MACD_Signal": signal_line,
    "MACD_Hist": macd_hist,
    "Volatility": volatility.values,
    "MACD_V_Line": macd_v_line.values,
    "MACD_V_Signal": macd_v_signal.values,
    "MACD_V_Hist": macd_v_hist.values
})

print("All MACD-V indicators calculated!")
print(f"Data shape: {indicators_df.shape}")
indicators_df.tail(5)

---
## Section 3: Signal Generation & Visualization

In [ ]:
# VISUALIZE MACD-V SIGNALS (default parameters)

# Entry: histogram crosses from negative to positive
entry_signals = (macd_v_hist.shift(1) < 0) & (macd_v_hist > 0)
# Exit: histogram crosses from positive to negative
exit_signals = (macd_v_hist.shift(1) > 0) & (macd_v_hist < 0)

print(f"Default MACD-V(12,26,9, vol=20) Signals:")
print(f"  Total Entry Signals: {entry_signals.sum()}")
print(f"  Total Exit Signals:  {exit_signals.sum()}")

# Plot
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 12), sharex=True,
                                     gridspec_kw={'height_ratios': [3, 1, 1]})

# Price with signals
ax1.plot(stock_data.index, close_arr, color='black', linewidth=1, label='Close')
entry_dates = stock_data.index[entry_signals.fillna(False).values]
exit_dates = stock_data.index[exit_signals.fillna(False).values]
ax1.scatter(entry_dates, close_series[entry_signals.fillna(False)], 
            marker='^', color='green', s=60, zorder=5, label='Long Entry')
ax1.scatter(exit_dates, close_series[exit_signals.fillna(False)], 
            marker='v', color='red', s=60, zorder=5, label='Long Exit')
ax1.set_title(f'{TICKER} Price with MACD-V Signals', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# MACD-V Line & Signal
ax2.plot(stock_data.index, macd_v_line, color='blue', linewidth=1, label='MACD-V Line')
ax2.plot(stock_data.index, macd_v_signal, color='orange', linewidth=1, label='MACD-V Signal')
ax2.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax2.set_title('MACD-V Line & Signal', fontsize=12, fontweight='bold')
ax2.set_ylabel('MACD-V', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# MACD-V Histogram
colors = ['green' if v >= 0 else 'red' for v in macd_v_hist.fillna(0)]
ax3.bar(stock_data.index, macd_v_hist.fillna(0), color=colors, width=1.5, alpha=0.7)
ax3.axhline(0, color='black', linewidth=0.8)
ax3.set_title('MACD-V Histogram', fontsize=12, fontweight='bold')
ax3.set_ylabel('Histogram', fontsize=11)
ax3.set_xlabel('Date', fontsize=11)
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## Section 4: Prepare Price Series & Train/Validation Split

In [ ]:
# PREPARE PRICE SERIES

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

def select_close_series(df, ticker):
    """Extract close price series handling both single and multi-index columns."""
    if isinstance(df.columns, pd.MultiIndex):
        if ('Close', ticker) in df.columns:
            s = df[('Close', ticker)]
        else:
            cols = [c for c in df.columns if 'Close' in str(c)]
            if not cols:
                raise KeyError("Close not found")
            s = df[cols[0]]
    else:
        s = df['Close']
    return s.astype(float).squeeze()

close = select_close_series(stock_data, TICKER)
close.name = 'price'

# 60/40 in-sample / out-of-sample split
TRAIN_RATIO = 0.60
split_idx = int(len(close) * TRAIN_RATIO)
train_close = close.iloc[:split_idx].copy()
val_close = close.iloc[split_idx:].copy()

print(f"Data ready: train={train_close.index[0].date()} → {train_close.index[-1].date()} | "
      f"val={val_close.index[0].date()} → {val_close.index[-1].date()}")
print(f"Train samples: {len(train_close)} | Val samples: {len(val_close)}")

---
## Section 5: MACD-V Grid Search Parameter Optimization

### Parameter Ranges
- **fast_period:** [8, 10, 12, 14, 16]
- **slow_period:** [20, 24, 26, 28, 32]
- **signal_period:** [7, 9, 11, 13]
- **volatility_period:** [10, 15, 20, 25, 30]

### Strategy Logic
- **Entry (Long):** MACD-V histogram crosses above zero
- **Exit (Long):** MACD-V histogram crosses below zero

### FTMO Considerations
- Fees set to 0.1% (0.001) to conservatively account for spreads
- Max Drawdown tracked explicitly (FTMO limit: 10% daily / 5% max)

---

In [ ]:
# DEFINE PARAMETER RANGES FOR MACD-V GRID SEARCH

fast_periods = [8, 10, 12, 14, 16]
slow_periods = [20, 24, 26, 28, 32]
signal_periods = [7, 9, 11, 13]
vol_periods = [10, 15, 20, 25, 30]

# Generate all valid combinations (fast < slow)
macdv_combinations = []
for fast, slow, sig, vol in product(fast_periods, slow_periods, signal_periods, vol_periods):
    if fast < slow:  # MACD requires fast < slow
        macdv_combinations.append((fast, slow, sig, vol))

print(f"MACD-V Parameter Ranges:")
print(f"  Fast periods:       {fast_periods}")
print(f"  Slow periods:       {slow_periods}")
print(f"  Signal periods:     {signal_periods}")
print(f"  Volatility periods: {vol_periods}")
print(f"\nGenerated {len(macdv_combinations)} valid MACD-V combinations")
print(f"\n📋 First 10 combinations preview:")
for i, (f, s, sg, v) in enumerate(macdv_combinations[:10], 1):
    print(f"  {i:2d}. Fast:{f:2d} | Slow:{s:2d} | Signal:{sg:2d} | VolPeriod:{v:2d}")
if len(macdv_combinations) > 10:
    print(f"   ... and {len(macdv_combinations) - 10} more combinations")
print("\nReady to test all combinations!")

In [ ]:
# INITIALIZE RESULTS COLLECTION

grid_search_results = []

print("MACD-V Results Collection System Initialized")
print(f"   - Will test {len(macdv_combinations)} MACD-V combinations")
print(f"   - Results will be stored in 'grid_search_results' list")

metrics_to_collect = [
    # Strategy Parameters
    "fast_period", "slow_period", "signal_period", "vol_period",
    # Return Metrics
    "total_return", "annualized_return",
    # Risk-Adjusted
    "sharpe_ratio", "sortino_ratio",
    # Risk
    "max_drawdown", "volatility",
    # Trade Performance
    "win_rate", "total_trades", "avg_trade_duration",
    "profit_factor", "trades_per_year",
    # Win/Loss
    "avg_win_amount", "avg_loss_amount", "expectancy"
]

print(f"\nMetrics to collect for each combination:")
for i, metric in enumerate(metrics_to_collect, 1):
    print(f"  {i}. {metric.replace('_', ' ').title()}")

print("\nReady to start the MACD-V grid search!")

In [ ]:
# HELPER FUNCTION: Calculate MACD-V signals for given parameters on a price series

def compute_macdv_signals(price_series, fast, slow, signal, vol_period):
    """
    Compute MACD-V entry/exit signals.
    
    Returns:
        entries (pd.Series[bool]): True where histogram crosses above zero
        exits (pd.Series[bool]): True where histogram crosses below zero
    """
    arr = price_series.values.astype(float)
    
    # Step 1: Traditional MACD via TA-Lib
    macd_line, _, _ = talib.MACD(arr, fastperiod=fast, slowperiod=slow, signalperiod=signal)
    
    # Step 2: Rolling volatility of returns (EWM std)
    rets = price_series.pct_change()
    vol = rets.ewm(span=vol_period).std()
    
    # Step 3: Normalize MACD by volatility
    macd_v = pd.Series(macd_line, index=price_series.index) / vol
    
    # Step 4: Signal on normalized MACD-V
    macd_v_sig = macd_v.ewm(span=signal).mean()
    
    # Step 5: Histogram
    hist = macd_v - macd_v_sig
    
    # Signal generation: histogram zero crossover
    entries_raw = (hist.shift(1) < 0) & (hist > 0)
    exits_raw = (hist.shift(1) > 0) & (hist < 0)
    
    # Shift by 1 bar to avoid lookahead bias (trade on NEXT bar after signal)
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    
    return entries, exits

print("✓ MACD-V signal computation function defined.")

In [ ]:
# MACD-V GRID SEARCH ON TRAINING DATA

print("INITIATING MACD-V GRID SEARCH OPTIMIZATION")
print("=" * 70)
print(f"Testing Strategy: MACD-V (Volatility-Normalized MACD)")
print(f"Training Period: {train_close.index[0].date()} → {train_close.index[-1].date()}")
print(f"Initial Capital: $100,000")
print(f"Transaction Costs: 0.1% per trade (FTMO conservative spread estimate)")
print(f"Optimization Metric: Sharpe Ratio (risk-adjusted returns)")
print("=" * 70)

total_combinations = len(macdv_combinations)
BATCH_SIZE = 100  # Smaller batches since MACD-V requires per-combo TA-Lib calls

print(f"Total combinations to test: {total_combinations}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Using vectorized backtesting for maximum speed\n")

grid_search_results = []
successful_tests = 0
failed_tests = 0

print("Starting batched grid search...\n")

for batch_start in range(0, total_combinations, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, total_combinations)
    batch_combos = macdv_combinations[batch_start:batch_end]
    batch_size = len(batch_combos)
    
    batch_num = batch_start // BATCH_SIZE + 1
    total_batches = (total_combinations + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"Processing batch {batch_num}/{total_batches} "
          f"(combos {batch_start+1} to {batch_end})...")
    
    # Build signal matrices for this batch
    batch_entries = []
    batch_exits = []
    valid_combos = []
    
    for fast, slow, sig, vol in batch_combos:
        try:
            entries, exits = compute_macdv_signals(train_close, fast, slow, sig, vol)
            batch_entries.append(entries)
            batch_exits.append(exits)
            valid_combos.append((fast, slow, sig, vol))
        except Exception as e:
            failed_tests += 1
    
    if not valid_combos:
        print(f"  ⚠️ No valid combinations in batch.")
        continue
    
    # Convert to DataFrames for vectorized backtesting
    entries_df = pd.DataFrame(batch_entries).T
    entries_df.index = train_close.index
    exits_df = pd.DataFrame(batch_exits).T
    exits_df.index = train_close.index
    
    # Vectorized backtest for entire batch at once
    try:
        portfolios = vbt.Portfolio.from_signals(
            close=train_close,
            entries=entries_df,
            exits=exits_df,
            init_cash=100_000,
            fees=0.001,       # 0.1% for FTMO spread estimate
            slippage=0.0005,
            freq='D'
        )
        
        # Extract batch metrics
        total_returns = portfolios.total_return()
        annualized_returns = portfolios.annualized_return(freq='D')
        max_drawdowns = portfolios.max_drawdown()
        volatilities = portfolios.annualized_volatility(freq='D')
        sharpe_ratios = portfolios.sharpe_ratio(freq='D')
        sortino_ratios = portfolios.sortino_ratio(freq='D')
        
        for idx, (fast, slow, sig, vol) in enumerate(valid_combos):
            try:
                total_return = float(total_returns.iloc[idx] if hasattr(total_returns, 'iloc') else total_returns)
                annualized_return = float(annualized_returns.iloc[idx] if hasattr(annualized_returns, 'iloc') else annualized_returns)
                max_drawdown = float(max_drawdowns.iloc[idx] if hasattr(max_drawdowns, 'iloc') else max_drawdowns)
                vol_ann = float(volatilities.iloc[idx] if hasattr(volatilities, 'iloc') else volatilities)
                sharpe_ratio = float(sharpe_ratios.iloc[idx] if hasattr(sharpe_ratios, 'iloc') else sharpe_ratios)
                sortino_ratio = float(sortino_ratios.iloc[idx] if hasattr(sortino_ratios, 'iloc') else sortino_ratios)
                
                # Trade-level metrics
                trades = portfolios[idx].trades if len(valid_combos) > 1 else portfolios.trades
                total_trades = len(trades)
                
                years = max((train_close.index[-1] - train_close.index[0]).days / 365.25, 1e-9)
                trades_per_year = total_trades / years
                
                # Minimum trade filter
                if trades_per_year < 1.5:
                    continue
                
                # Trade statistics
                win_rate_pct = np.nan
                profit_factor = np.nan
                expectancy = 0.0
                avg_win_amount = 0.0
                avg_loss_amount = 0.0
                avg_duration = np.nan
                
                if total_trades > 0:
                    tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
                    if tr.size > 0:
                        pos = tr[tr > 0]
                        neg = tr[tr < 0]
                        win_rate_pct = (len(pos) / len(tr)) * 100.0 if len(tr) > 0 else np.nan
                        gains = pos.sum() if len(pos) else 0.0
                        losses = abs(neg.sum()) if len(neg) else 0.0
                        profit_factor = gains / losses if losses > 0 else np.inf
                        expectancy = float(tr.mean())
                        avg_win_amount = float(pos.mean()) if len(pos) else 0.0
                        avg_loss_amount = float(abs(neg.mean())) if len(neg) else 0.0
                    
                    # Average trade duration
                    try:
                        durations = trades.duration
                        if hasattr(durations, 'mean'):
                            avg_duration = float(durations.mean().total_seconds() / 86400) if hasattr(durations.mean(), 'total_seconds') else float(durations.mean())
                    except:
                        pass
                
                grid_search_results.append({
                    "fast_period": fast,
                    "slow_period": slow,
                    "signal_period": sig,
                    "vol_period": vol,
                    "total_return": total_return,
                    "annualized_return": annualized_return,
                    "max_drawdown": max_drawdown,
                    "volatility": vol_ann,
                    "sharpe_ratio": sharpe_ratio,
                    "sortino_ratio": sortino_ratio,
                    "total_trades": total_trades,
                    "win_rate": win_rate_pct,
                    "profit_factor": profit_factor,
                    "expectancy": expectancy,
                    "avg_win_amount": avg_win_amount,
                    "avg_loss_amount": avg_loss_amount,
                    "avg_trade_duration": avg_duration,
                    "trades_per_year": trades_per_year
                })
                successful_tests += 1
                
            except Exception as e:
                failed_tests += 1
    
    except Exception as e:
        print(f"  ⚠️ Batch failed: {str(e)[:100]}")
        failed_tests += batch_size
    
    progress_pct = (batch_end / total_combinations) * 100
    print(f"  ✓ Progress: {batch_end}/{total_combinations} ({progress_pct:.1f}%) | "
          f"Successful: {successful_tests}")

# SUMMARY
print("\n" + "=" * 70)
print("GRID SEARCH COMPLETED!")
print("=" * 70)
print(f"Total combinations attempted: {total_combinations}")
print(f"Successfully completed: {successful_tests}")
print(f"Failed/filtered: {failed_tests + (total_combinations - successful_tests - failed_tests)}")
print(f"\n✓ Results stored in 'grid_search_results' ({len(grid_search_results)} entries)")

if successful_tests > 0:
    results_df_preview = pd.DataFrame(grid_search_results)
    
    print("\n" + "=" * 70)
    print("🏆 TOP 5 COMBINATIONS (by In-Sample Sharpe Ratio)")
    print("=" * 70)
    
    top_5 = results_df_preview.nlargest(5, 'sharpe_ratio')
    for rank, (idx, row) in enumerate(top_5.iterrows(), 1):
        print(f"\n#{rank} - MACD-V(F={int(row['fast_period'])}, S={int(row['slow_period'])}, "
              f"Sig={int(row['signal_period'])}, Vol={int(row['vol_period'])})")
        print(f"   Sharpe Ratio:      {row['sharpe_ratio']:.3f}")
        print(f"   Total Return:      {row['total_return']:.2%}")
        print(f"   Annualized Return: {row['annualized_return']:.2%}")
        print(f"   Max Drawdown:      {row['max_drawdown']:.2%}")
        print(f"   Win Rate:          {row['win_rate']:.1f}%")
        print(f"   Profit Factor:     {row['profit_factor']:.2f}")
        print(f"   Total Trades:      {int(row['total_trades'])} ({row['trades_per_year']:.1f}/year)")
    
    print("\n" + "=" * 70)

---
## Section 6: Results Display & Analysis

In [ ]:
# GRID SEARCH RESULTS ANALYSIS

results_df = pd.DataFrame(grid_search_results)

print("Grid Search Results Analysis")
print("=" * 50)
print(f"Total combinations tested: {len(results_df)}")
print(f"Results shape: {results_df.shape}")

print("\nComprehensive Performance Statistics:")
print("-" * 50)
print("Return Metrics:")
print(f"   Best Total Return: {results_df['total_return'].max():.2%}")
print(f"   Average Total Return: {results_df['total_return'].mean():.2%}")
print(f"   Best Annualized Return: {results_df['annualized_return'].max():.2%}")
print("Risk-Adjusted Return Metrics:")
print(f"   Best Sharpe Ratio: {results_df['sharpe_ratio'].max():.3f}")
print(f"   Median Sharpe Ratio: {results_df['sharpe_ratio'].median():.3f}")
print(f"   Best Sortino Ratio: {results_df['sortino_ratio'].max():.3f}")
print("Risk Metrics:")
print(f"   Average Max Drawdown: {results_df['max_drawdown'].mean():.2%}")
print(f"   Best (smallest) Max DD: {results_df['max_drawdown'].max():.2%}")
print("Trade Performance:")
print(f"   Best Win Rate: {results_df['win_rate'].max():.1f}%")
print(f"   Average Win Rate: {results_df['win_rate'].mean():.1f}%")
print(f"   Best Profit Factor: {results_df['profit_factor'].replace([np.inf], np.nan).max():.2f}")
print(f"   Avg Trades/Year: {results_df['trades_per_year'].mean():.1f}")

# Display top 50 sorted by Sharpe
print("\n" + "=" * 70)
print("TOP 50 PARAMETER COMBINATIONS (sorted by IS Sharpe Ratio)")
print("=" * 70)

display_cols = ['fast_period', 'slow_period', 'signal_period', 'vol_period',
                'sharpe_ratio', 'total_return', 'annualized_return', 'max_drawdown',
                'win_rate', 'profit_factor', 'total_trades', 'trades_per_year']
top_50 = results_df.nlargest(50, 'sharpe_ratio')[display_cols].copy()
top_50['total_return'] = top_50['total_return'].apply(lambda x: f"{x:.2%}")
top_50['annualized_return'] = top_50['annualized_return'].apply(lambda x: f"{x:.2%}")
top_50['max_drawdown'] = top_50['max_drawdown'].apply(lambda x: f"{x:.2%}")
top_50['sharpe_ratio'] = top_50['sharpe_ratio'].apply(lambda x: f"{x:.3f}")
top_50['win_rate'] = top_50['win_rate'].apply(lambda x: f"{x:.1f}%")
top_50['profit_factor'] = top_50['profit_factor'].apply(lambda x: f"{x:.2f}")
print(top_50.to_string(index=False))

---
## Section 7: Out-of-Sample Validation & IS vs OOS Comparison

Run the top parameter combinations on the out-of-sample (validation) data to check for robustness.

In [ ]:
# OUT-OF-SAMPLE VALIDATION FOR TOP COMBINATIONS

results_df_raw = pd.DataFrame(grid_search_results)

# Take top 50 IS combinations to validate OOS
top_n = min(50, len(results_df_raw))
top_is = results_df_raw.nlargest(top_n, 'sharpe_ratio').copy()

oos_results = []

print(f"Validating top {top_n} IS combinations on OOS data...")
print(f"OOS Period: {val_close.index[0].date()} → {val_close.index[-1].date()}\n")

for _, row in top_is.iterrows():
    fast = int(row['fast_period'])
    slow = int(row['slow_period'])
    sig = int(row['signal_period'])
    vol = int(row['vol_period'])
    
    try:
        entries_oos, exits_oos = compute_macdv_signals(val_close, fast, slow, sig, vol)
        
        pf_oos = vbt.Portfolio.from_signals(
            close=val_close,
            entries=entries_oos,
            exits=exits_oos,
            init_cash=100_000,
            fees=0.001,
            slippage=0.0005,
            freq='D'
        )
        
        oos_sharpe = float(pf_oos.sharpe_ratio(freq='D'))
        oos_return = float(pf_oos.total_return())
        oos_maxdd = float(pf_oos.max_drawdown())
        oos_trades = len(pf_oos.trades)
        
        # Trade stats
        oos_win_rate = np.nan
        oos_profit_factor = np.nan
        if oos_trades > 0:
            tr = pf_oos.trades.returns.values
            pos = tr[tr > 0]
            neg = tr[tr < 0]
            oos_win_rate = (len(pos) / len(tr)) * 100.0
            gains = pos.sum() if len(pos) else 0.0
            losses = abs(neg.sum()) if len(neg) else 0.0
            oos_profit_factor = gains / losses if losses > 0 else np.inf
        
        oos_results.append({
            'fast_period': fast, 'slow_period': slow,
            'signal_period': sig, 'vol_period': vol,
            'is_sharpe': float(row['sharpe_ratio']),
            'is_return': float(row['total_return']),
            'is_maxdd': float(row['max_drawdown']),
            'is_win_rate': float(row['win_rate']),
            'is_trades': int(row['total_trades']),
            'oos_sharpe': oos_sharpe,
            'oos_return': oos_return,
            'oos_maxdd': oos_maxdd,
            'oos_win_rate': oos_win_rate,
            'oos_profit_factor': oos_profit_factor,
            'oos_trades': oos_trades
        })
    except Exception as e:
        pass

oos_df = pd.DataFrame(oos_results).sort_values('is_sharpe', ascending=False)

print("\n" + "=" * 90)
print("IS vs OOS COMPARISON TABLE")
print("=" * 90)
print("\nInterpretation: OOS Sharpe within 30% of IS Sharpe = ROBUST")
print("               FTMO Max DD: Must stay under 10% daily\n")

# Format and display
display_oos = oos_df.copy()
display_oos['sharpe_decay'] = ((display_oos['oos_sharpe'] - display_oos['is_sharpe']) / 
                                display_oos['is_sharpe'].abs() * 100)
display_oos['robust'] = display_oos['sharpe_decay'].abs() < 30

for col in ['is_sharpe', 'oos_sharpe']:
    display_oos[col] = display_oos[col].apply(lambda x: f"{x:.3f}")
for col in ['is_return', 'oos_return', 'is_maxdd', 'oos_maxdd']:
    display_oos[col] = display_oos[col].apply(lambda x: f"{x:.2%}")
display_oos['sharpe_decay'] = display_oos['sharpe_decay'].apply(lambda x: f"{x:.1f}%")
display_oos['robust'] = display_oos['robust'].apply(lambda x: '✅' if x else '❌')

print(display_oos[['fast_period', 'slow_period', 'signal_period', 'vol_period',
                    'is_sharpe', 'is_return', 'is_maxdd',
                    'oos_sharpe', 'oos_return', 'oos_maxdd', 'oos_trades',
                    'sharpe_decay', 'robust']].head(30).to_string(index=False))

robust_count = sum(1 for r in oos_results 
                   if abs((r['oos_sharpe'] - r['is_sharpe']) / max(abs(r['is_sharpe']), 1e-9)) < 0.3)
print(f"\n✅ Robust combinations (OOS within 30% of IS): {robust_count}/{len(oos_results)}")

---
## Section 8: Sensitivity Analysis (CRITICAL — Rick's Framework)

For each parameter, vary it ±5 steps while holding all others at the **base** (best IS Sharpe) values.  
Compute Sharpe delta % to identify fragile vs. robust parameters.

**Color Code:**
- 🟢 Dark Green: Δ > +10%  
- 🟢 Light Green: 0% < Δ < +10%  
- 🟠 Orange: -10% < Δ < 0%  
- 🔴 Red: Δ < -10%

---

In [ ]:
# SENSITIVITY ANALYSIS

results_df = pd.DataFrame(grid_search_results)
FREQ = '1D'

# Identify base parameters (best IS Sharpe)
best_row = results_df.loc[results_df['sharpe_ratio'].idxmax()]
fast_base = int(best_row['fast_period'])
slow_base = int(best_row['slow_period'])
signal_base = int(best_row['signal_period'])
vol_base = int(best_row['vol_period'])

print(f"📊 BASE PARAMETERS (Best IS Sharpe):")
print(f"   Fast={fast_base}, Slow={slow_base}, Signal={signal_base}, VolPeriod={vol_base}")
print(f"   IS Sharpe: {best_row['sharpe_ratio']:.3f}")
print(f"   IS Return: {best_row['total_return']:.2%}")
print(f"   IS MaxDD:  {best_row['max_drawdown']:.2%}")

# Generate variation ranges (±5 around base)
fast_range = list(range(max(2, fast_base - 5), fast_base + 6))
slow_range = list(range(max(fast_base + 2, slow_base - 5), slow_base + 6))
signal_range = list(range(max(2, signal_base - 5), signal_base + 6))
vol_range = list(range(max(3, vol_base - 5), vol_base + 6))

print(f"\n📐 Variation Ranges:")
print(f"   Fast:   {fast_range}")
print(f"   Slow:   {slow_range}")
print(f"   Signal: {signal_range}")
print(f"   Vol:    {vol_range}")

# Build all sensitivity combos
all_combos = set()
for f in fast_range:
    if f < slow_base:
        all_combos.add((f, slow_base, signal_base, vol_base))
for s in slow_range:
    if fast_base < s:
        all_combos.add((fast_base, s, signal_base, vol_base))
for sg in signal_range:
    all_combos.add((fast_base, slow_base, sg, vol_base))
for v in vol_range:
    all_combos.add((fast_base, slow_base, signal_base, v))

all_combos = sorted(list(all_combos))

# Evaluate each combo on both IS and OOS
def eval_combo_both(fast, slow, sig, vol):
    """Run backtest on both IS and OOS for a single parameter set."""
    entries_is, exits_is = compute_macdv_signals(train_close, fast, slow, sig, vol)
    pf_is = vbt.Portfolio.from_signals(
        close=train_close, entries=entries_is, exits=exits_is,
        init_cash=100_000, fees=0.001, slippage=0.0005, freq=FREQ
    )
    
    entries_oos, exits_oos = compute_macdv_signals(val_close, fast, slow, sig, vol)
    pf_oos = vbt.Portfolio.from_signals(
        close=val_close, entries=entries_oos, exits=exits_oos,
        init_cash=100_000, fees=0.001, slippage=0.0005, freq=FREQ
    )
    
    return {
        'fast': fast, 'slow': slow, 'signal': sig, 'vol': vol,
        'is_sharpe': float(pf_is.sharpe_ratio(freq='D')),
        'is_return': float(pf_is.total_return()),
        'is_maxdd': float(pf_is.max_drawdown()),
        'oos_sharpe': float(pf_oos.sharpe_ratio(freq='D')),
        'oos_return': float(pf_oos.total_return()),
        'oos_maxdd': float(pf_oos.max_drawdown()),
        'oos_trades': len(pf_oos.trades)
    }

print(f"\n🔄 Testing {len(all_combos)} parameter variations...")

rows = []
for combo in all_combos:
    try:
        rows.append(eval_combo_both(*combo))
    except Exception:
        pass

if not rows:
    print("❌ No sensitivity results computed.")
else:
    sens_df = pd.DataFrame(rows)
    base_is_sharpe = float(best_row['sharpe_ratio'])
    
    # Get OOS sharpe for base
    base_oos_row = sens_df[(sens_df['fast'] == fast_base) & (sens_df['slow'] == slow_base) &
                           (sens_df['signal'] == signal_base) & (sens_df['vol'] == vol_base)]
    base_oos_sharpe = float(base_oos_row['oos_sharpe'].iloc[0]) if len(base_oos_row) > 0 else np.nan
    
    print(f"   Base IS Sharpe:  {base_is_sharpe:.3f}")
    print(f"   Base OOS Sharpe: {base_oos_sharpe:.3f}")
    
    # Split by parameter variation type
    fast_variations = sens_df[(sens_df['slow'] == slow_base) & (sens_df['signal'] == signal_base) & 
                              (sens_df['vol'] == vol_base)].copy().sort_values('fast')
    slow_variations = sens_df[(sens_df['fast'] == fast_base) & (sens_df['signal'] == signal_base) & 
                              (sens_df['vol'] == vol_base)].copy().sort_values('slow')
    signal_variations = sens_df[(sens_df['fast'] == fast_base) & (sens_df['slow'] == slow_base) & 
                                (sens_df['vol'] == vol_base)].copy().sort_values('signal')
    vol_variations = sens_df[(sens_df['fast'] == fast_base) & (sens_df['slow'] == slow_base) & 
                             (sens_df['signal'] == signal_base)].copy().sort_values('vol')
    
    # Calculate deltas
    for var_df in [fast_variations, slow_variations, signal_variations, vol_variations]:
        var_df['is_sharpe_delta'] = ((var_df['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100)
        if not np.isnan(base_oos_sharpe) and abs(base_oos_sharpe) > 1e-6:
            var_df['oos_sharpe_delta'] = ((var_df['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100)
        else:
            var_df['oos_sharpe_delta'] = np.nan
    
    # ============================================================
    # CREATE BAR CHARTS — 8-PANEL LAYOUT (4 params × IS/OOS)
    # ============================================================
    fig, axes = plt.subplots(2, 4, figsize=(26, 10))
    fig.suptitle(f'Parameter Sensitivity Analysis — Base: MACD-V(F={fast_base}, S={slow_base}, '
                 f'Sig={signal_base}, Vol={vol_base})',
                 fontsize=16, fontweight='bold')
    
    param_configs = [
        ('Fast Period', fast_variations, 'fast', fast_base),
        ('Slow Period', slow_variations, 'slow', slow_base),
        ('Signal Period', signal_variations, 'signal', signal_base),
        ('Vol Period', vol_variations, 'vol', vol_base),
    ]
    
    for col_idx, (param_name, var_df, param_col, base_val) in enumerate(param_configs):
        if var_df.empty:
            continue
        
        # IS row
        colors_is = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen'
                     for x in var_df['is_sharpe_delta']]
        axes[0, col_idx].bar(var_df[param_col], var_df['is_sharpe_delta'],
                             color=colors_is, edgecolor='black', linewidth=0.5)
        axes[0, col_idx].axhline(0, color='black', linewidth=1.5)
        axes[0, col_idx].axvline(base_val, color='blue', linestyle='--', linewidth=2.5,
                                 alpha=0.7, label=f'Base={base_val}')
        axes[0, col_idx].set_xlabel(param_name, fontsize=11, fontweight='bold')
        axes[0, col_idx].set_ylabel('Sharpe Δ%', fontsize=11, fontweight='bold')
        axes[0, col_idx].set_title(f'{param_name} — In-Sample', fontsize=12, fontweight='bold')
        axes[0, col_idx].grid(axis='y', alpha=0.3)
        axes[0, col_idx].legend(fontsize=9)
        
        # OOS row
        if not np.isnan(base_oos_sharpe) and 'oos_sharpe_delta' in var_df.columns:
            oos_deltas = var_df['oos_sharpe_delta'].fillna(0)
            colors_oos = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen'
                         for x in oos_deltas]
            axes[1, col_idx].bar(var_df[param_col], oos_deltas,
                                 color=colors_oos, edgecolor='black', linewidth=0.5)
            axes[1, col_idx].axhline(0, color='black', linewidth=1.5)
            axes[1, col_idx].axvline(base_val, color='blue', linestyle='--', linewidth=2.5,
                                     alpha=0.7, label=f'Base={base_val}')
            axes[1, col_idx].set_xlabel(param_name, fontsize=11, fontweight='bold')
            axes[1, col_idx].set_ylabel('Sharpe Δ%', fontsize=11, fontweight='bold')
            axes[1, col_idx].set_title(f'{param_name} — Out-of-Sample', fontsize=12, fontweight='bold')
            axes[1, col_idx].grid(axis='y', alpha=0.3)
            axes[1, col_idx].legend(fontsize=9)
        else:
            axes[1, col_idx].text(0.5, 0.5, 'OOS N/A', transform=axes[1, col_idx].transAxes,
                                  ha='center', va='center', fontsize=14)
            axes[1, col_idx].set_title(f'{param_name} — Out-of-Sample', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # ============================================================
    # COMPACT SENSITIVITY SUMMARY TABLE
    # ============================================================
    print("\n📋 SENSITIVITY SUMMARY")
    print("=" * 90)
    
    summary_data = []
    for param_name, variations, param_col in [
        ('Fast', fast_variations, 'fast'),
        ('Slow', slow_variations, 'slow'),
        ('Signal', signal_variations, 'signal'),
        ('VolPeriod', vol_variations, 'vol')
    ]:
        if variations.empty:
            continue
        is_max_delta = variations['is_sharpe_delta'].abs().max()
        oos_max_delta_val = variations['oos_sharpe_delta'].abs().max() if 'oos_sharpe_delta' in variations and not variations['oos_sharpe_delta'].isna().all() else np.nan
        
        summary_data.append({
            'Parameter': param_name,
            'IS Range': f"{variations['is_sharpe'].min():.3f} — {variations['is_sharpe'].max():.3f}",
            'IS Max Δ%': f"{variations['is_sharpe_delta'].min():.1f}%",
            'OOS Range': (f"{variations['oos_sharpe'].min():.3f} — {variations['oos_sharpe'].max():.3f}"
                          if not np.isnan(base_oos_sharpe) else 'N/A'),
            'OOS Max Δ%': (f"{variations['oos_sharpe_delta'].min():.1f}%"
                           if not np.isnan(base_oos_sharpe) and 'oos_sharpe_delta' in variations
                           else 'N/A'),
            'Sensitivity': '⚠️ HIGH' if is_max_delta > 40 else '✅ LOW'
        })
    
    summary_table = pd.DataFrame(summary_data)
    print(summary_table.to_string(index=False))
    
    print("\n✅ Analysis Complete! Green = Robust, Red = Fragile")
    print("   LOW sensitivity = preferred for live FTMO trading")

---
## Section 9: Best Strategy Summary & Equity Curves

Display the final selected strategy with full IS/OOS metrics and equity curve visualization.

In [ ]:
# BEST STRATEGY SUMMARY WITH EQUITY CURVES

results_df = pd.DataFrame(grid_search_results)
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
fast_best = int(best['fast_period'])
slow_best = int(best['slow_period'])
signal_best = int(best['signal_period'])
vol_best = int(best['vol_period'])

print("=" * 70)
print("🏆 BEST MACD-V STRATEGY SUMMARY")
print("=" * 70)
print(f"\nParameters: Fast={fast_best}, Slow={slow_best}, Signal={signal_best}, VolPeriod={vol_best}")

# Run on full data for equity curve
entries_full, exits_full = compute_macdv_signals(close, fast_best, slow_best, signal_best, vol_best)

pf_full = vbt.Portfolio.from_signals(
    close=close,
    entries=entries_full,
    exits=exits_full,
    init_cash=100_000,
    fees=0.001,
    slippage=0.0005,
    freq='D'
)

# IS metrics
entries_is, exits_is = compute_macdv_signals(train_close, fast_best, slow_best, signal_best, vol_best)
pf_is = vbt.Portfolio.from_signals(
    close=train_close, entries=entries_is, exits=exits_is,
    init_cash=100_000, fees=0.001, slippage=0.0005, freq='D'
)

# OOS metrics
entries_oos, exits_oos = compute_macdv_signals(val_close, fast_best, slow_best, signal_best, vol_best)
pf_oos = vbt.Portfolio.from_signals(
    close=val_close, entries=entries_oos, exits=exits_oos,
    init_cash=100_000, fees=0.001, slippage=0.0005, freq='D'
)

print(f"\n{'Metric':<25} {'In-Sample':>15} {'Out-of-Sample':>15}")
print("-" * 55)
print(f"{'Sharpe Ratio':<25} {pf_is.sharpe_ratio(freq='D'):>15.3f} {pf_oos.sharpe_ratio(freq='D'):>15.3f}")
print(f"{'Sortino Ratio':<25} {pf_is.sortino_ratio(freq='D'):>15.3f} {pf_oos.sortino_ratio(freq='D'):>15.3f}")
print(f"{'Total Return':<25} {pf_is.total_return():>14.2%} {pf_oos.total_return():>14.2%}")
print(f"{'Max Drawdown':<25} {pf_is.max_drawdown():>14.2%} {pf_oos.max_drawdown():>14.2%}")
print(f"{'Total Trades':<25} {len(pf_is.trades):>15d} {len(pf_oos.trades):>15d}")

# Win rate
for label, pf in [('IS', pf_is), ('OOS', pf_oos)]:
    tr = pf.trades.returns.values
    wr = (len(tr[tr > 0]) / len(tr) * 100) if len(tr) > 0 else 0
    if label == 'IS':
        print(f"{'Win Rate':<25} {wr:>14.1f}%", end='')
    else:
        print(f" {wr:>14.1f}%")

sharpe_decay = ((pf_oos.sharpe_ratio(freq='D') - pf_is.sharpe_ratio(freq='D')) / 
                abs(pf_is.sharpe_ratio(freq='D')) * 100)
print(f"\nSharpe Decay IS→OOS: {sharpe_decay:.1f}%")
print(f"Robustness: {'✅ ROBUST' if abs(sharpe_decay) < 30 else '⚠️ FRAGILE'}")

# ============================================================
# EQUITY CURVE PLOT (IS vs OOS periods)
# ============================================================
ret_full = pf_full.returns()
eq_full = (1 + ret_full).cumprod() * 100_000

split_date = train_close.index[-1]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

# Equity Curve
ax1.plot(eq_full.index, eq_full.values, color='black', linewidth=1.5, label='Equity Curve')
ax1.axvline(split_date, color='red', linestyle='--', linewidth=2, alpha=0.8, label='IS/OOS Split')
ax1.fill_between(eq_full.index, eq_full.values, alpha=0.1, color='blue')

# Mark IS and OOS regions
ax1.axvspan(eq_full.index[0], split_date, alpha=0.05, color='green', label='In-Sample')
ax1.axvspan(split_date, eq_full.index[-1], alpha=0.05, color='orange', label='Out-of-Sample')

ax1.set_title(f'MACD-V(F={fast_best}, S={slow_best}, Sig={signal_best}, Vol={vol_best}) — '
              f'Equity Curve (IS vs OOS)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Portfolio Value ($)', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# Drawdown subplot
peak = eq_full.cummax()
dd = (eq_full - peak) / peak
ax2.fill_between(dd.index, dd.values * 100, color='red', alpha=0.4)
ax2.axvline(split_date, color='red', linestyle='--', linewidth=2, alpha=0.8)
ax2.set_ylabel('Drawdown %', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.set_title('Underwater Equity (Drawdowns)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================
# MACD-V INDICATOR SUBPLOT WITH SIGNALS ON PRICE
# ============================================================
print("\n--- MACD-V Signal Overlay on Price ---")

# Recalculate MACD-V for full period for visualization
arr_full = close.values.astype(float)
macd_full, _, _ = talib.MACD(arr_full, fastperiod=fast_best, slowperiod=slow_best, signalperiod=signal_best)
rets_full = close.pct_change()
vol_full = rets_full.ewm(span=vol_best).std()
macdv_full = pd.Series(macd_full, index=close.index) / vol_full
macdv_sig_full = macdv_full.ewm(span=signal_best).mean()
macdv_hist_full = macdv_full - macdv_sig_full

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 12), sharex=True,
                                     gridspec_kw={'height_ratios': [3, 1, 1]})

# Price + signals
ax1.plot(close.index, close.values, color='black', linewidth=1, label='Close')
entry_mask = entries_full.fillna(False)
exit_mask = exits_full.fillna(False)
ax1.scatter(close.index[entry_mask], close[entry_mask], marker='^', color='green',
            s=50, zorder=5, label='Long Entry')
ax1.scatter(close.index[exit_mask], close[exit_mask], marker='v', color='red',
            s=50, zorder=5, label='Long Exit')
ax1.axvline(split_date, color='red', linestyle='--', linewidth=2, alpha=0.7, label='IS/OOS Split')
ax1.set_title(f'{TICKER} Price with MACD-V Signals (Best Parameters)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# MACD-V line & signal
ax2.plot(close.index, macdv_full, color='blue', linewidth=1, label='MACD-V')
ax2.plot(close.index, macdv_sig_full, color='orange', linewidth=1, label='Signal')
ax2.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax2.axvline(split_date, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax2.set_ylabel('MACD-V', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# Histogram
hist_colors = ['green' if v >= 0 else 'red' for v in macdv_hist_full.fillna(0)]
ax3.bar(close.index, macdv_hist_full.fillna(0), color=hist_colors, width=1.5, alpha=0.7)
ax3.axhline(0, color='black', linewidth=0.8)
ax3.axvline(split_date, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax3.set_ylabel('Histogram', fontsize=11)
ax3.set_xlabel('Date', fontsize=11)
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ MACD-V Strategy Notebook Complete!")
print("\nNext Steps:")
print("  1. Test on FTMO assets: EURUSD, GBPUSD, USDJPY (forex)")
print("  2. Test on indices: US30, NAS100, SPX500")
print("  3. Test on gold: XAUUSD")
print("  4. Adapt TICKER variable and re-run entire notebook")